# 01 — Data Exploration

**Environmental Impact Analysis — EVAT T1 2026**

This notebook explores the four datasets used in this use case:

| Dataset | Description |
|---------|-------------|
| `ev_vehicles.csv` | 58 EV models with WLTP energy consumption |
| `ice_vehicles.csv` | 65 ICE vehicles with WLTP fuel consumption |
| `grid_emission_factors.csv` | State-level Australian grid intensity (DCCEEW 2023) |
| `fuel_emission_factors.csv` | Fuel type CO2 emission factors (DCCEEW 2023) |

---

In [ ]:
import sys
from pathlib import Path

# Add project root to path so src/ is importable
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

from src.data_loader import load_ev_vehicles, load_ice_vehicles, load_grid_factors, load_fuel_factors
from src.config import GRID_EMISSION_FACTORS, FUEL_EMISSION_FACTORS

print('Imports OK')

## 1. EV Vehicle Database

In [ ]:
ev = load_ev_vehicles()
print(f'Shape: {ev.shape}')
ev.head(10)

In [ ]:
print('EV Consumption (kWh/100km) statistics:')
print(ev['Consumption_kWh_per_100km'].describe().round(2))
print(f'\nNumber of unique makes: {ev["Make"].nunique()}')
print(f'Makes: {sorted(ev["Make"].unique().tolist())}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of EV consumption
axes[0].hist(ev['Consumption_kWh_per_100km'], bins=15, color='#2196F3', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('WLTP Consumption (kWh/100km)')
axes[0].set_ylabel('Count')
axes[0].set_title('EV Energy Consumption Distribution')
axes[0].axvline(ev['Consumption_kWh_per_100km'].mean(), color='red', linestyle='--', label=f'Mean: {ev["Consumption_kWh_per_100km"].mean():.1f}')
axes[0].legend()

# Average consumption by body style
avg_by_style = ev.groupby('BodyStyle')['Consumption_kWh_per_100km'].mean().sort_values()
axes[1].barh(avg_by_style.index, avg_by_style.values, color='#2196F3')
axes[1].set_xlabel('Average WLTP Consumption (kWh/100km)')
axes[1].set_title('Average EV Consumption by Body Style')
for i, v in enumerate(avg_by_style.values):
    axes[1].text(v + 0.1, i, f'{v:.1f}', va='center')

plt.tight_layout()
plt.show()

## 2. ICE Vehicle Database

In [ ]:
ice = load_ice_vehicles()
print(f'Shape: {ice.shape}')
print(f'\nFuel types: {ice["FuelType"].value_counts().to_dict()}')
ice.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Consumption by fuel type
for fuel, color in [('Petrol', '#FF5722'), ('Diesel', '#607D8B')]:
    subset = ice[ice['FuelType'] == fuel]['Consumption_Combined_L100km']
    axes[0].hist(subset, bins=12, alpha=0.7, label=fuel, color=color, edgecolor='white')
axes[0].set_xlabel('WLTP Combined Consumption (L/100km)')
axes[0].set_ylabel('Count')
axes[0].set_title('ICE Fuel Consumption Distribution by Fuel Type')
axes[0].legend()

# Average by segment
avg_seg = ice.groupby('Segment')['Consumption_Combined_L100km'].mean().sort_values()
axes[1].barh(avg_seg.index, avg_seg.values, color='#FF5722')
axes[1].set_xlabel('Average WLTP Consumption (L/100km)')
axes[1].set_title('Average ICE Consumption by Segment')
for i, v in enumerate(avg_seg.values):
    axes[1].text(v + 0.05, i, f'{v:.1f}', va='center')

plt.tight_layout()
plt.show()

## 3. Grid Emission Factors — Why State Matters

This is the most critical factor the previous model got wrong. The original model used **0.18 kg CO₂/kWh** nationally — which is South Australia's figure (dominated by wind and solar). Most Australians live in NSW, VIC, and QLD where the grid is far dirtier.

In [ ]:
grid = load_grid_factors()
grid[['State', 'State_Full', 'Grid_Intensity_g_per_kWh', 'Notes']]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

states = grid['State']
intensities = grid['Grid_Intensity_g_per_kWh']
colors = ['#4CAF50' if v < 300 else '#FF9800' if v < 700 else '#F44336' for v in intensities]

bars = ax.bar(states, intensities, color=colors, edgecolor='white', linewidth=0.5)

# Add the old model's value
ax.axhline(180, color='purple', linestyle='--', linewidth=1.5, label='Old model (hardcoded): 180 g/kWh')

for bar, val in zip(bars, intensities):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f'{val}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Grid Emission Intensity (g CO₂/kWh)')
ax.set_title('Australian Grid Emission Intensity by State (DCCEEW 2023)\nGreen = clean  |  Orange = moderate  |  Red = coal-heavy')
ax.legend()
ax.set_ylim(0, 1100)

plt.tight_layout()
plt.show()

print(f'\nRatio VIC/SA: {990/290:.1f}x  — VIC grid is {990/290:.1f}x dirtier than SA')
print(f'Old model error for VIC: used {180} g/kWh, actual {990} g/kWh ({990/180:.1f}x underestimate)')

## 4. Impact of State on CO₂ Savings

Demonstrating how the same EV–ICE pair produces very different savings depending on which state the driver lives in.

In [ ]:
from src.calculator import ev_co2_g_per_km, ice_co2_g_per_km

# Tesla Model 3 SR vs Toyota Corolla Petrol
ev_kwh = 14.9
ice_l = 6.0
ice_fuel = 'Petrol'

rows = []
for state, intensity_kg in GRID_EMISSION_FACTORS.items():
    ev_co2 = ev_co2_g_per_km(ev_kwh, state)
    ice_co2 = ice_co2_g_per_km(ice_l, ice_fuel)
    savings = ice_co2 - ev_co2
    rows.append({
        'State': state,
        'Grid (g/kWh)': int(intensity_kg * 1000),
        'EV (g CO₂/km)': round(ev_co2, 1),
        'ICE (g CO₂/km)': round(ice_co2, 1),
        'Saving (g/km)': round(savings, 1),
        'Saving (%)': round(savings / ice_co2 * 100, 1),
    })

df_state = pd.DataFrame(rows).sort_values('Saving (g/km)', ascending=False)
print('Tesla Model 3 SR (14.9 kWh/100km) vs Toyota Corolla Petrol (6.0 L/100km):')
print(df_state.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

savings_vals = df_state.set_index('State')['Saving (g/km)']
colors = ['#4CAF50' if v > 50 else '#FF9800' if v > 0 else '#F44336' for v in savings_vals]

bars = ax.bar(savings_vals.index, savings_vals.values, color=colors)
ax.axhline(0, color='black', linewidth=0.8)

for bar, val in zip(bars, savings_vals.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (1 if val >= 0 else -4),
            f'{val:.0f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=9)

ax.set_ylabel('CO₂ Saving (g per km)')
ax.set_title('Tesla Model 3 vs Toyota Corolla — CO₂ Saving by Australian State\n(WLTP, no lifecycle, DCCEEW 2023 grid factors)')

plt.tight_layout()
plt.show()

## 5. Lifecycle vs Operational Emissions

Including battery manufacturing emissions shows a more honest picture, especially in coal-heavy states.

In [ ]:
from src.calculator import calculate

print('With lifecycle (battery manufacturing) included:')
print('=' * 65)

for state in ['SA', 'TAS', 'NSW', 'VIC', 'QLD']:
    result = calculate(
        ev_consumption_kwh_per_100km=14.9,
        ev_battery_kwh=57.5,
        ice_consumption_l_per_100km=6.0,
        ice_fuel_type='Petrol',
        state=state,
        include_lifecycle=True,
    )
    print(f'{state}: EV {result.ev_total_g_per_km:.1f} g/km '
          f'(op {result.ev_operational_g_per_km:.1f} + mfg {result.ev_manufacturing_g_per_km:.1f}) '
          f'vs ICE {result.ice_operational_g_per_km:.1f} g/km '
          f'→ saving {result.co2_savings_g_per_km:.1f} g/km ({result.percentage_reduction:.0f}%)')

---

**Key findings from this EDA:**

1. **State grid intensity is the dominant variable.** SA (290 g/kWh) vs VIC (990 g/kWh) — a 3.4× difference — completely changes the CO₂ story.
2. **The old model's 180 g/kWh national average drastically overstated savings** in NSW, VIC, and QLD (home to ~70% of Australians).
3. **Lifecycle emissions (battery manufacturing) add ~8–20 g/km** to the EV footprint. In coal-heavy states this can significantly reduce — or even eliminate — the advantage over an efficient petrol car.
4. **ICE segment matters.** A large diesel SUV/ute vs a small EV hatch yields very different savings than comparing equivalently-sized vehicles.

Proceed to `02_Model_Training_Evaluation.ipynb` to train the XGBoost real-world adjustment model and see the full pipeline in action.